# Akan-BPE — Qwen3-0.6B × Akan TTS tokenizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/run-qwen-0.6b.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/run-qwen-0.6b.ipynb)

QLoRA fine-tune of a base LLM with the Akan TTS tokenizer, scored against the unmodified base on **bits-per-byte (BPB, full byte coverage)**. Runs two arms — random and mean-of-subword embedding init — and compares them. Run top-to-bottom on a Kaggle/Colab **T4 GPU** (~45–50 min per arm).

## Setup

In [ ]:
# Clone the repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

In [ ]:
!pip install -q uv
!uv pip install -q --system -e ".[dev,train]"
!uv pip install -q --system bitsandbytes sentencepiece

In [ ]:
# Hugging Face authentication. Paste a token when prompted — it unblocks and
# speeds up the dataset downloads below (input is hidden).
from getpass import getpass

from huggingface_hub import login

HF_TOKEN = getpass("Enter your Hugging Face token: ").strip()
login(token=HF_TOKEN)

In [ ]:
# Confirm a GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable a T4 GPU accelerator, then re-run.")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Download Akan datasets. --tts-limit 50000 pins the 45,000/2,500/2,500 split.
!python scripts/download.py --output-dir data --tts-limit 50000

In [ ]:
# Train the TTS tokenizer if not already present
from pathlib import Path

if not Path("models/tts_tokenizer.json").exists():
    !python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts
else:
    print("TTS tokenizer already present, skipping.")

In [ ]:
# Verify required inputs exist
from pathlib import Path

required = {
    "TTS train data": Path("data/pristine_twi_train.jsonl"),
    "TTS test data": Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer": Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")

## Run — random vs mean-of-subword embedding init

Defaults (eval cap, max-length, LoRA rank, learning rate, etc.) live in `scripts/model_integration.py`; pass `--help` to see or override them. Each arm writes its own adapter dir + result JSON derived from the model id.

In [ ]:
# Arm A — random embedding init (default)
!python scripts/model_integration.py --model-id Qwen/Qwen3-0.6B-Base

In [ ]:
# Arm B — mean-of-subword embedding init
!python scripts/model_integration.py --model-id Qwen/Qwen3-0.6B-Base --embedding-init-mode mean_subword

## Results — compare the two arms

In [ ]:
import json
from pathlib import Path

def load_eval(path):
    r = json.loads(Path(path).read_text(encoding="utf-8"))
    b = r["eval"]["bpb"]
    return {
        "init": r["embedding_init_mode"],
        "eval_loss": r["eval"]["eval_loss"],
        "perplexity": r["eval"]["perplexity"],
        "exp_bpb": b["experiment"]["bits_per_byte"],
        "base_bpb": (b["base"] or {}).get("bits_per_byte"),
        "improvement": b.get("improvement"),
    }

arms = {
    "random": load_eval("results/run-qwen-0.6b.json"),
    "mean_subword": load_eval("results/run-qwen-0.6b-meansub.json"),
}
header = f"{'metric':<24}{'random':>14}{'mean_subword':>16}"
print(header)
print("-" * len(header))

def show(label, key, fmt="{:.4f}"):
    cells = [fmt.format(arms[a][key]) if arms[a][key] is not None else "—"
             for a in ("random", "mean_subword")]
    print(f"{label:<24}{cells[0]:>14}{cells[1]:>16}")

show("eval_loss", "eval_loss")
show("perplexity", "perplexity", "{:.2f}")
show("experiment BPB", "exp_bpb")
show("base BPB", "base_bpb")
show("improvement (base−exp)", "improvement")

best = min(arms, key=lambda a: arms[a]["exp_bpb"])
print(f"\nLower fine-tuned BPB: {best!r} arm.")